# Deduplication Analysis
This notebook verifies the results of the deduplication and aggregation process performed on the merged TCR AnnData dataset.

In [1]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import scirpy as ir


# Display settings
pd.set_option('display.max_colwidth', 100)

## Loading the Deduplicated AnnData Dataset
We load the deduplicated dataset from the cache. We also attempt to load the merged (pre-deduplication) dataset to compare sizes.

In [2]:
cache_dir = "/cluster/home/dego/.cache/iggytop_anndata"
dedup_path = f"{cache_dir}/deduplicated_anndata.h5ad"
merged_path = f"{cache_dir}/merged_anndata.h5ad"

adata_dedup = sc.read_h5ad(dedup_path)
print(f"Loaded deduplicated data: {adata_dedup.shape}")

try:
    adata_merged = sc.read_h5ad(merged_path)
    print(f"Loaded merged data: {adata_merged.shape}")
except FileNotFoundError:
    print("Merged data not found")


/raid/persistent_scratch/dego/uv/venvs/iggytop/lib/python3.11/site-packages/anndata/utils.py:362: ExperimentalFeatureWarning: Support for Awkward Arrays is currently experimental. Behavior may change in the future. Please report any issues you may encounter!
  warnings.warn(msg, category, stacklevel=stacklevel)


Loaded deduplicated data: (336613, 0)
Loaded merged data: (549347, 0)


### Data Dimension and Statistics Analysis
We calculate the deduplication ratio $R = \frac{N_{unique}}{N_{total}}$ to see how much redundancy was removed.

In [3]:
n_total = adata_merged.n_obs
n_unique = adata_dedup.n_obs
ratio = n_unique / n_total

print(f"Total entries: {n_total:,}")
print(f"Unique entries: {n_unique:,}")
print(f"Deduplication Ratio R: {ratio:.4f}")
print(f"Reduction: {(1-ratio)*100:.2f}%")

Total entries: 549,347
Unique entries: 336,613
Deduplication Ratio R: 0.6128
Reduction: 38.72%


In [7]:
adata_dedup

AnnData object with n_obs × n_vars = 336613 × 0
    obs: 'MHC_class', 'MHC_gene_1', 'PMID', 'antigen_name', 'antigen_species', 'epitope_sequence', 'iedb_iri', 'tissue', 'source'
    obsm: 'airr', 'chain_indices'

In [3]:
adata_merged

AnnData object with n_obs × n_vars = 549347 × 0
    obs: 'MHC_class', 'MHC_gene_1', 'PMID', 'antigen_name', 'antigen_species', 'epitope_sequence', 'iedb_iri', 'tissue', 'source'
    obsm: 'airr', 'chain_indices'

## Inverstigate MHC data sources and availability

In [4]:
nans = adata_merged.obs['MHC_class'] == 'nan'
nans_sum = nans.sum()
print(f"Number of 'nan' entries in 'MHC_class': {nans_sum}")
sources_with_nan_mhc = adata_merged.obs.loc[adata_merged.obs['MHC_class'] == 'nan', 'source'].unique()
print(f"Sources with 'nan' in MHC_class:\n{sources_with_nan_mhc}")

Number of 'nan' entries in 'MHC_class': 12143
Sources with 'nan' in MHC_class:
['McPAS', 'IEDB', 'NeoTCR', 'CEDAR', 'TRAIT']
Categories (7, object): ['VDJdb', 'McPAS', 'IEDB', 'TCR3d', 'NeoTCR', 'CEDAR', 'TRAIT']


In [5]:
nans = adata_merged.obs['MHC_gene_1'] == 'nan'
nans_sum = nans.sum()
print(f"Number of 'nan' entries in 'MHC_gene_1': {nans_sum}")
sources_with_nan_mhc = adata_merged.obs.loc[adata_merged.obs['MHC_gene_1'] == 'nan', 'source'].unique()
print(f"Sources with 'nan' in MHC_gene_1:\n{sources_with_nan_mhc}")

Number of 'nan' entries in 'MHC_gene_1': 12149
Sources with 'nan' in MHC_gene_1:
['McPAS', 'IEDB', 'TCR3d', 'NeoTCR', 'CEDAR', 'TRAIT']
Categories (7, object): ['VDJdb', 'McPAS', 'IEDB', 'TCR3d', 'NeoTCR', 'CEDAR', 'TRAIT']


In [6]:
# Identify rows where MHC_gene_1 is available but MHC_class is 'nan'
mask = (adata_merged.obs['MHC_gene_1'] != 'nan') & (adata_merged.obs['MHC_class'] == 'nan')
inconsistent_rows = adata_merged.obs[mask]

print(f"Number of rows where MHC_gene_1 is available but MHC_class is 'nan': {len(inconsistent_rows)}")
print("this is expected as we can infer the MHC class from the gene name")

Number of rows where MHC_gene_1 is available but MHC_class is 'nan': 0
this is expected as we can infer the MHC class from the gene name


In [7]:
source_counts = adata_merged.obs['source'].value_counts()
print(source_counts)

# Count and print 'nan' values in 'source'
nans_in_source = adata_merged.obs['source'].isna().sum()
print(f"\nNumber of 'nan' entries in 'source': {nans_in_source}")

source
IEDB      234055
VDJdb     139442
CEDAR     106869
TRAIT      51524
McPAS      16199
NeoTCR       887
TCR3d        371
Name: count, dtype: int64

Number of 'nan' entries in 'source': 0


In [8]:
source_counts = adata_dedup.obs['source'].value_counts()
print(source_counts)

# Count and print 'nan' values in 'source'
nans_in_source = adata_dedup.obs['source'].isna().sum()
print(f"\nNumber of 'nan' entries in 'source': {nans_in_source}")

source
IEDB                                   116306
CEDAR|IEDB                              85001
VDJdb                                   67296
TRAIT|VDJdb                             27731
CEDAR|IEDB|TRAIT|VDJdb                  10669
McPAS                                    9151
TRAIT                                    6884
CEDAR|IEDB|McPAS                         2902
CEDAR|IEDB|VDJdb                         2670
IEDB|VDJdb                               1997
IEDB|TRAIT|VDJdb                         1973
CEDAR|IEDB|McPAS|TRAIT|VDJdb             1880
NeoTCR                                    443
McPAS|TRAIT|VDJdb                         348
McPAS|VDJdb                               219
NeoTCR|TRAIT|VDJdb                        158
CEDAR|IEDB|NeoTCR|TRAIT|VDJdb             151
IEDB|McPAS                                129
TCR3d                                     106
CEDAR|IEDB|NeoTCR                          95
CEDAR|IEDB|McPAS|VDJdb                     70
CEDAR|IEDB|TRAIT           

as we already knew, cedar is part of IEDB and thus gets removed during deduplication

## 3.5. Investigating Missing Junction Sequences
The deduplication process warned about `NaN` values in `VJ_1_junction_aa` and `VDJ_1_junction_aa`. This section identifies those rows and their sources.

In [10]:

# We use airr_context to access the junction columns properly
with ir.get.airr_context(adata_merged, ["junction_aa"], chain=["VJ_1", "VDJ_1"]) as m:
    df_check = m.obs.copy()
    # In the merged data, missing AIRR entries are represented as None/NaN
    vj_missing = df_check[df_check['VJ_1_junction_aa'].isna()]
    vdj_missing = df_check[df_check['VDJ_1_junction_aa'].isna()]
    
    print(f"Total rows: {len(df_check)}")
    print(f"Rows with empty VJ_1_junction_aa: {len(vj_missing)} ({len(vj_missing)/len(df_check):.2%})")
    print(f"Rows with empty VDJ_1_junction_aa: {len(vdj_missing)} ({len(vdj_missing)/len(df_check):.2%})")
    
    print("\nSource distribution for rows with missing VJ junctions:")
    print(vj_missing['source'].value_counts().head())
    
    print("\nSource distribution for rows with missing VDJ junctions:")
    print(vdj_missing['source'].value_counts().head())

    # Example of rows where BOTH are missing
    both_missing = df_check[(df_check['VJ_1_junction_aa'].isna()) & (df_check['VDJ_1_junction_aa'].isna())]
    print(f"\nRows where BOTH junctions are missing: {len(both_missing)}")
    
    if len(both_missing) > 0:
        print('sources where both junctions are missing:')
        print(both_missing['source'].value_counts().head())
        print("\nExamples of rows with both junctions missing:")
        display(both_missing[['source', 'PMID', 'iedb_iri']].head())

Total rows: 549347
Rows with empty VJ_1_junction_aa: 285367 (51.95%)
Rows with empty VDJ_1_junction_aa: 77092 (14.03%)

Source distribution for rows with missing VJ junctions:
source
IEDB     162510
CEDAR     55141
VDJdb     37014
TRAIT     20229
McPAS      9972
Name: count, dtype: int64

Source distribution for rows with missing VDJ junctions:
source
IEDB     31337
CEDAR    24770
VDJdb    15376
TRAIT     4452
McPAS     1141
Name: count, dtype: int64

Rows where BOTH junctions are missing: 0


In [11]:
# We use airr_context to access the junction columns properly
with ir.get.airr_context(adata_dedup, ["junction_aa"], chain=["VJ_1", "VDJ_1"]) as m:
    df_check = m.obs.copy()
    # In the merged data, missing AIRR entries are represented as None/NaN
    vj_missing = df_check[df_check['VJ_1_junction_aa'].isna()]
    vdj_missing = df_check[df_check['VDJ_1_junction_aa'].isna()]
    
    print(f"Total deduplicated rows: {len(df_check)}")
    print(f"Rows with empty VJ_1_junction_aa: {len(vj_missing)} ({len(vj_missing)/len(df_check):.2%})")
    print(f"Rows with empty VDJ_1_junction_aa: {len(vdj_missing)} ({len(vdj_missing)/len(df_check):.2%})")
    
    print("\nSource distribution for rows with missing VJ junctions:")
    print(vj_missing['source'].value_counts().head())
    
    print("\nSource distribution for rows with missing VDJ junctions:")
    print(vdj_missing['source'].value_counts().head())

    # Example of rows where BOTH are missing
    both_missing = df_check[(df_check['VJ_1_junction_aa'].isna()) & (df_check['VDJ_1_junction_aa'].isna())]
    print(f"\nRows where BOTH junctions are missing: {len(both_missing)}")
    
    if len(both_missing) > 0:
        print('sources where both junctions are missing:')
        print(both_missing['source'].value_counts().head())
        print("\nExamples of rows with both junctions missing:")
        display(both_missing[['source', 'PMID', 'iedb_iri']].head())

Total deduplicated rows: 336613
Rows with empty VJ_1_junction_aa: 180191 (53.53%)
Rows with empty VDJ_1_junction_aa: 36272 (10.78%)

Source distribution for rows with missing VJ junctions:
source
IEDB                      102254
CEDAR|IEDB                 42970
TRAIT|VDJdb                 9601
VDJdb                       7094
CEDAR|IEDB|TRAIT|VDJdb      5194
Name: count, dtype: int64

Source distribution for rows with missing VDJ junctions:
source
CEDAR|IEDB                20391
VDJdb                      4850
IEDB                       4721
CEDAR|IEDB|TRAIT|VDJdb     3207
TRAIT|VDJdb                 817
Name: count, dtype: int64

Rows where BOTH junctions are missing: 0


Explore PMID col

In [16]:
pmid_with_prefix = adata_dedup.obs['PMID'].str.contains('https', regex=False).sum()
print(f"Number of entries containing 'https': {pmid_with_prefix}")

Number of entries containing 'https': 30894


In [15]:
pmid_with_prefix = adata_dedup.obs['PMID'].str.contains('|', regex=False).sum()
print(f"Number of entries containing '|': {pmid_with_prefix}")

Number of entries containing '|': 6150


In [14]:
pmid_with_prefix = adata_dedup.obs['PMID'].str.contains('nan', regex=False).sum()
print(f"Number of entries containing 'nan': {pmid_with_prefix}")

Number of entries containing 'nan': 293


In [18]:
adata_merged.obs['PMID'][adata_merged.obs['PMID'].str.contains('nan')]

cell_id
86379_VDJdb    nan
86380_VDJdb    nan
86381_VDJdb    nan
86382_VDJdb    nan
86383_VDJdb    nan
              ... 
366_TCR3d      nan
367_TCR3d      nan
368_TCR3d      nan
369_TCR3d      nan
370_TCR3d      nan
Name: PMID, Length: 925, dtype: category
Categories (3604, object): ['1313573', '1381757', '1517580', '1522584', ..., 'nan', 'no_pmid_1024523', 'no_pmid_1036521', 'unpublished']

## 4. Inspecting Metadata for Deduplication Flags
We check the `PMID` and `source` columns for aggregated values (those containing the `|` separator). This shows which records were found in multiple databases or publications.

In [21]:
# Find records with multiple sources
multi_source = adata_dedup.obs[adata_dedup.obs['source'].str.contains('|', regex=False)]
print(f"Number of records with multiple sources: {len(multi_source)}")

# Find records with multiple PMIDs
multi_pmid = adata_dedup.obs[adata_dedup.obs['PMID'].str.contains('|', regex=False)]
print(f"Number of records with multiple PMIDs: {len(multi_pmid)}")


Number of records with multiple sources: 136426
Number of records with multiple PMIDs: 6150


## 5. Post-Deduplication Integrity Verification
We verify that the resulting AnnData has unique observation names and perform a quick check on the columns.

In [22]:
is_unique = adata_dedup.obs_names.is_unique
print(f"Observation names are unique: {is_unique}")

if not is_unique:
    print("Warning: Duplicates found in index!")

print("\nColumns in the final dataset:")
print(adata_dedup.obs.columns.tolist())

Observation names are unique: True

Columns in the final dataset:
['MHC_class', 'MHC_gene_1', 'PMID', 'antigen_name', 'antigen_species', 'epitope_sequence', 'iedb_iri', 'tissue', 'source']
